In [1]:
import scanpy as sc
import numpy as np
import pandas as pd
from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\annam\Dissertation 2026")
PROCESSED_DIR = PROJECT_DIR / "Data" / "Processed"
RESULTS_DIR = PROJECT_DIR / "results" / "phase2_clustering_v2"

adata1 = sc.read_h5ad(PROCESSED_DIR / "GSE114725_phase2_v2_annotated.h5ad")
print(f"Loaded: {adata1.n_obs} cells")
print(adata1.obs["cell_type"].value_counts())

Loaded: 44662 cells
cell_type
T cells                                               20213
CD8/Effector T cells                                   7861
Macrophages                                            5914
NK/Cytotoxic T cells                                   4929
B cells                                                3369
Mixed/stromal-contaminated (CD8+fibroblast signal)     1335
Mast cells                                              494
Monocytes/DC                                            359
pDC                                                     188
Name: count, dtype: int64


In [2]:
# ----------------------------
# Step 1 — basic marker check on the existing "NK/Cytotoxic T cells"
# cluster, per Adrien's comment: true NK cells are CD3D/CD3E/TRAC
# negative but NCAM1/KLRF1/NCR1/FCGR3A positive, while cytotoxic CD8 T
# cells are CD3/TRAC positive and CD8A positive.
# ----------------------------
adata1_raw = adata1.raw.to_adata()
adata1_raw.obs["cell_type"] = adata1.obs["cell_type"].values

nk_mask = adata1.obs["cell_type"] == "NK/Cytotoxic T cells"
for gene in ["CD3D", "CD3E", "TRAC", "NCAM1", "KLRF1", "FCGR3A", "CD8A"]:
    if gene in adata1_raw.var_names:
        X = adata1_raw[nk_mask.values, gene].X
        if hasattr(X, "toarray"): X = X.toarray()
        pct_pos = (X > 0).mean() * 100
        print(f"{gene}: {pct_pos:.1f}% of cells positive")

CD3D: 19.9% of cells positive
CD3E: 19.5% of cells positive
NCAM1: 6.0% of cells positive
KLRF1: 26.6% of cells positive
FCGR3A: 46.7% of cells positive
CD8A: 28.1% of cells positive


In [3]:
# ----------------------------
# Step 2 — simple 2-way split: CD3D/CD3E positive = likely mislabeled
# cytotoxic CD8 T cells
# ----------------------------
nk_subset_raw = adata1_raw[nk_mask.values].copy()

cd3d = nk_subset_raw[:, "CD3D"].X
cd3e = nk_subset_raw[:, "CD3E"].X
if hasattr(cd3d, "toarray"): cd3d = cd3d.toarray().flatten()
if hasattr(cd3e, "toarray"): cd3e = cd3e.toarray().flatten()

is_t_cell_like = (cd3d > 0) | (cd3e > 0)

n_total = len(is_t_cell_like)
n_t_like = is_t_cell_like.sum()
n_true_nk = n_total - n_t_like

print(f"Total cells in 'NK/Cytotoxic T cells' cluster: {n_total}")
print(f"Likely mislabeled (CD3D or CD3E positive -> cytotoxic CD8 T cells): {n_t_like} ({n_t_like/n_total*100:.1f}%)")
print(f"Likely true NK cells (CD3D and CD3E negative): {n_true_nk} ({n_true_nk/n_total*100:.1f}%)")

cd8a = nk_subset_raw[:, "CD8A"].X
if hasattr(cd8a, "toarray"): cd8a = cd8a.toarray().flatten()

pct_cd8a_in_mislabeled = (cd8a[is_t_cell_like] > 0).mean() * 100
pct_cd8a_in_true_nk = (cd8a[~is_t_cell_like] > 0).mean() * 100

print(f"\nCD8A positive among likely-mislabeled cells: {pct_cd8a_in_mislabeled:.1f}%")
print(f"CD8A positive among likely-true-NK cells: {pct_cd8a_in_true_nk:.1f}%")

Total cells in 'NK/Cytotoxic T cells' cluster: 4929
Likely mislabeled (CD3D or CD3E positive -> cytotoxic CD8 T cells): 1591 (32.3%)
Likely true NK cells (CD3D and CD3E negative): 3338 (67.7%)

CD8A positive among likely-mislabeled cells: 46.6%
CD8A positive among likely-true-NK cells: 19.4%


In [4]:
# ----------------------------
# Step 3 — 3-way split, per Adrien's follow-up suggestion: check
# whether T-cell-marker-positive cells ALSO co-express NK markers,
# indicating genuine NKT cells rather than simply mislabeled CD8.
# ----------------------------
def get_expr(adata, gene):
    if gene not in adata.var_names:
        return None
    X = adata[:, gene].X
    if hasattr(X, "toarray"):
        X = X.toarray()
    return X.flatten()

ncam1 = get_expr(nk_subset_raw, "NCAM1")
klrf1 = get_expr(nk_subset_raw, "KLRF1")
fcgr3a = get_expr(nk_subset_raw, "FCGR3A")

t_cell_marker = is_t_cell_like
nk_marker = (ncam1 > 0) | (klrf1 > 0) | (fcgr3a > 0)

is_true_nk = ~t_cell_marker & nk_marker
is_nkt = t_cell_marker & nk_marker
is_cytotoxic_cd8 = t_cell_marker & ~nk_marker
is_unclear = ~t_cell_marker & ~nk_marker

print(f"True NK (T-marker neg, NK-marker pos):        {is_true_nk.sum():5d} ({is_true_nk.sum()/n_total*100:.1f}%)")
print(f"NKT cells (T-marker pos, NK-marker pos):       {is_nkt.sum():5d} ({is_nkt.sum()/n_total*100:.1f}%)")
print(f"Cytotoxic CD8 (T-marker pos, NK-marker neg):   {is_cytotoxic_cd8.sum():5d} ({is_cytotoxic_cd8.sum()/n_total*100:.1f}%)")
print(f"Unclear (both negative):                       {is_unclear.sum():5d} ({is_unclear.sum()/n_total*100:.1f}%)")

print("\nCD8A positivity by group:")
for name, mask in [("True NK", is_true_nk), ("NKT", is_nkt),
                    ("Cytotoxic CD8", is_cytotoxic_cd8), ("Unclear", is_unclear)]:
    if mask.sum() > 0:
        pct_cd8a = (cd8a[mask] > 0).mean() * 100
        print(f"  {name} (n={mask.sum()}): {pct_cd8a:.1f}% CD8A positive")

classification = pd.Series("Unclear", index=nk_subset_raw.obs_names)
classification[is_true_nk] = "True NK"
classification[is_nkt] = "NKT"
classification[is_cytotoxic_cd8] = "Cytotoxic CD8 (mislabeled)"
classification.to_csv(RESULTS_DIR / "GSE114725_NK_cluster_3way_classification.csv")
print(f"\nSaved classification to GSE114725_NK_cluster_3way_classification.csv")

True NK (T-marker neg, NK-marker pos):         2192 (44.5%)
NKT cells (T-marker pos, NK-marker pos):         748 (15.2%)
Cytotoxic CD8 (T-marker pos, NK-marker neg):     843 (17.1%)
Unclear (both negative):                        1146 (23.3%)

CD8A positivity by group:
  True NK (n=2192): 15.0% CD8A positive
  NKT (n=748): 47.6% CD8A positive
  Cytotoxic CD8 (n=843): 45.7% CD8A positive
  Unclear (n=1146): 27.7% CD8A positive

Saved classification to GSE114725_NK_cluster_3way_classification.csv


In [5]:
t_cell_genes = [g for g in ["CD3D", "CD3E"] if g in nk_subset_raw.var_names]
nk_genes = [g for g in ["NCAM1", "KLRF1", "FCGR3A"] if g in nk_subset_raw.var_names]

sc.tl.score_genes(nk_subset_raw, t_cell_genes, score_name="T_score")
sc.tl.score_genes(nk_subset_raw, nk_genes, score_name="NK_score")

t_cell_marker_v2 = nk_subset_raw.obs["T_score"] > 0
nk_marker_v2 = nk_subset_raw.obs["NK_score"] > 0

is_true_nk_v2 = ~t_cell_marker_v2 & nk_marker_v2
is_nkt_v2 = t_cell_marker_v2 & nk_marker_v2
is_cytotoxic_cd8_v2 = t_cell_marker_v2 & ~nk_marker_v2
is_unclear_v2 = ~t_cell_marker_v2 & ~nk_marker_v2

n_total = len(t_cell_marker_v2)
print(f"True NK:        {is_true_nk_v2.sum():5d} ({is_true_nk_v2.sum()/n_total*100:.1f}%)")
print(f"NKT cells:      {is_nkt_v2.sum():5d} ({is_nkt_v2.sum()/n_total*100:.1f}%)")
print(f"Cytotoxic CD8:  {is_cytotoxic_cd8_v2.sum():5d} ({is_cytotoxic_cd8_v2.sum()/n_total*100:.1f}%)")
print(f"Unclear:        {is_unclear_v2.sum():5d} ({is_unclear_v2.sum()/n_total*100:.1f}%)")

True NK:         2283 (46.3%)
NKT cells:        571 (11.6%)
Cytotoxic CD8:    705 (14.3%)
Unclear:         1370 (27.8%)


In [6]:
# ----------------------------
# Investigate the "Unclear" cells — three checks:
# 1. Are they just low-quality/low-depth cells (dropout hypothesis)?
# 2. Do they still show the core cytotoxic signal that put them in this
#    cluster in the first place (NKG7/GNLY/PRF1/GZMB)?
# 3. What genes actually ARE distinguishing them, if anything?
# ----------------------------

# Reconstruct the original hard-threshold grouping (23.3% version)
is_unclear = ~is_t_cell_like & ~nk_marker  # from earlier in this notebook

# Check 1 — sequencing depth comparison
adata1_full_qc = adata1.obs.loc[nk_subset_raw.obs_names]
n_genes = adata1_full_qc["n_genes_by_counts"].values
total_counts = adata1_full_qc["total_counts"].values

print("=== Sequencing depth by group ===")
for name, mask in [("True NK", is_true_nk), ("NKT", is_nkt),
                    ("Cytotoxic CD8", is_cytotoxic_cd8), ("Unclear", is_unclear)]:
    print(f"  {name} (n={mask.sum()}): "
          f"mean n_genes={n_genes[mask].mean():.0f}, "
          f"mean total_counts={total_counts[mask].mean():.0f}")

# Check 2 — do Unclear cells still show core cytotoxic signal?
print("\n=== Core cytotoxic markers in Unclear cells (should be positive, since") 
print("=== that's why they're in this cluster at all) ===")
for gene in ["NKG7", "GNLY", "PRF1", "GZMB"]:
    expr = get_expr(nk_subset_raw, gene)
    if expr is not None:
        pct_pos_unclear = (expr[is_unclear] > 0).mean() * 100
        pct_pos_overall = (expr > 0).mean() * 100
        print(f"  {gene}: {pct_pos_unclear:.1f}% positive in Unclear (vs {pct_pos_overall:.1f}% overall)")

# Check 3 — top marker genes specifically for Unclear vs everyone else
nk_subset_raw.obs["group_3way"] = "Other"
nk_subset_raw.obs.loc[nk_subset_raw.obs_names[is_unclear], "group_3way"] = "Unclear"
nk_subset_raw.obs["group_3way"] = nk_subset_raw.obs["group_3way"].astype("category")

sc.tl.rank_genes_groups(nk_subset_raw, groupby="group_3way", groups=["Unclear"],
                        reference="Other", method="wilcoxon")
top_genes_unclear = sc.get.rank_genes_groups_df(nk_subset_raw, group="Unclear").head(15)
print("\n=== Top genes distinguishing Unclear cells from the rest ===")
print(top_genes_unclear[["names", "logfoldchanges", "pvals_adj"]].to_string(index=False))

=== Sequencing depth by group ===
  True NK (n=2192): mean n_genes=510, mean total_counts=933
  NKT (n=748): mean n_genes=586, mean total_counts=1052
  Cytotoxic CD8 (n=843): mean n_genes=524, mean total_counts=944
  Unclear (n=1146): mean n_genes=442, mean total_counts=804

=== Core cytotoxic markers in Unclear cells (should be positive, since
=== that's why they're in this cluster at all) ===
  NKG7: 85.7% positive in Unclear (vs 89.6% overall)
  GNLY: 89.9% positive in Unclear (vs 92.0% overall)
  PRF1: 74.6% positive in Unclear (vs 82.6% overall)
  GZMB: 46.6% positive in Unclear (vs 55.9% overall)

=== Top genes distinguishing Unclear cells from the rest ===
 names  logfoldchanges  pvals_adj
 RPS18        0.151317   0.012131
RPL13A        0.110067   0.013195
 RPS29        0.145190   0.016045
 RPL30        0.195581   0.062304
 RPS27        0.081217   0.092506
RPS27A        0.071730   0.196668
RPL27A        0.057706   0.232439
  RPL3        0.086572   0.414846
 RPL28        0.089567